<a href="https://colab.research.google.com/github/ImVrishank/VecStreetBoys/blob/main/Word2Vec(16th_july).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install datasets torch.nn torch pyarrow -q

ERROR: Could not find a version that satisfies the requirement torch.nn (from versions: none)
ERROR: No matching distribution found for torch.nn


In [23]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import re
from collections import Counter
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
splits = {'test': 'wikitext-103-raw-v1/test-00000-of-00001.parquet', 'train': 'wikitext-103-raw-v1/train-*.parquet', 'validation': 'wikitext-103-raw-v1/validation-00000-of-00001.parquet'}
ddf = dd.read_parquet("hf://datasets/Salesforce/wikitext/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [25]:
df = ddf.compute()
display(df.head())

,text
0,
1,= Robert Boulter =
2,
3,"Robert Boulter is an English film , televisio..."
4,"In 2006 , Boulter starred alongside Whishaw i..."


Cleaning up this data:

- remove all spl char
- make all rows into a singular corpus of text
- add the unk keyword to extra words
-

In [26]:
# Splitting corpus into words

corpus = df['text'].str.cat(sep=' ')
print(corpus[:500])

  = Robert Boulter = 
   Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens , which was performed in 2001 at the Royal Court Theatre . He had a guest role in the television series Judge John Deed in 2002 . In 2004 Boulter landed a role as " Craig " in the episode " Teddy 's Story " of the television series The Long Firm ; h


In [27]:
# Removing all special chars

corpus = re.sub(r'[^a-zA-Z0-9\s]', '', corpus)
print(corpus[:500])

   Robert Boulter  
   Robert Boulter is an English film  television and theatre actor  He had a guest  starring role on the television series The Bill in 2000  This was followed by a starring role in the play Herons written by Simon Stephens  which was performed in 2001 at the Royal Court Theatre  He had a guest role in the television series Judge John Deed in 2002  In 2004 Boulter landed a role as  Craig  in the episode  Teddy s Story  of the television series The Long Firm  he starred alongsi


In [28]:
# Removing Capitalization

corpus = corpus.lower()
print(corpus[:500])

   robert boulter  
   robert boulter is an english film  television and theatre actor  he had a guest  starring role on the television series the bill in 2000  this was followed by a starring role in the play herons written by simon stephens  which was performed in 2001 at the royal court theatre  he had a guest role in the television series judge john deed in 2002  in 2004 boulter landed a role as  craig  in the episode  teddy s story  of the television series the long firm  he starred alongsi


In [29]:
# Building the vocabulary (top 5000 words in order of most used)

words = corpus.split()
word_counts = Counter(words)

vocabulary = [word for word, count in word_counts.most_common(5000)]

print(vocabulary[:100])

['the', 'of', 'and', 'in', 'to', 'a', 'was', 'on', 'as', 'that', 'for', 's', 'with', 'by', 'he', 'at', 'his', 'is', 'were', 'from', 'it', 'had', 'an', 'which', 'are', 'be', 'first', 'also', 'this', 'but', 'its', 'after', 'their', 'not', 'two', 'one', 'they', 'have', 'been', 'or', '1', 'during', 'has', 'lesnar', 'when', 'into', 'war', 'time', 'would', 'who', 'all', 'more', 'new', 'only', 'nero', 'north', 'most', 'other', 'three', 'while', 'city', 'him', '2', 'out', 'american', 'tropical', 'there', 'these', 'division', 'later', 'no', 'some', '5', 'film', 'manila', 'up', '3', 'may', 'world', 'made', 'over', 'through', 'year', 'head', 'both', 'her', 'than', 'about', 'british', 'several', 'under', 'before', '000', 'km', 'between', 'game', 'season', 'number', 'against', 'used']


In [ ]:
# Removing all words that arent in vocabulary and replacing with <unk>

words = corpus.split()
processed_words = [word if word in vocabulary else '<unk>' for word in words]
processed_corpus = ' '.join(processed_words)

print(processed_corpus[:500])

robert boulter robert boulter is an english film television and theatre actor he had a guest starring role on the television series the bill in 2000 this was followed by a starring role in the play <unk> written by simon <unk> which was performed in 2001 at the royal court theatre he had a guest role in the television series judge john <unk> in 2002 in 2004 boulter landed a role as craig in the episode <unk> s story of the television series the long <unk> he starred alongside actors mark strong 


In [ ]:
# Function to make feature and label vectors

def get_database(corpus, context_len):
  words = corpus.split()
  data = []

  for pointer in range(context_len, len(words) - context_len):
    context = words[pointer - context_len:pointer] + words[pointer + 1:pointer + context_len]
    lable = words[pointer]

    data.append({'context': ' '.join(context), 'label': lable})

  df = pd.DataFrame(data)
  return df



In [ ]:
# Context_window = 5, takes 5 words from before the present and 5 words after the present word
context_window_size = 5
context_df = get_database(processed_corpus, context_window_size)

print(context_df.head())

                                             context       label
0  robert boulter robert boulter is english film ...          an
1  boulter robert boulter is an film television a...     english
2  robert boulter is an english television and th...        film
3    boulter is an english film and theatre actor he  television
4  is an english film television theatre actor he...         and


# One hot Encodings

Now that we have the Context and Label. We need to make one hot vectors for each word. We need to add the one hot vectors element-wise to get the feature vectors and keep the label vector as it is.


# Splitting the Dataset
I needed to split to the dataset into four quarters because:
- I kept running out of memory
- It took a long time, so i did not want to end up losing the entire progress by mistake



In [ ]:
# first quarter

dummy_data = {"Features": np.zeros(len(vocabulary)), "Label": np.zeros(len(vocabulary))}
final_df = pd.DataFrame(dummy_data)
pd.set_option('display.max_colwidth', None)

for i in range(context_df.shape[0] // 4):
  bag_of_words = context_df['context'][i].split()
  label = context_df['label'][i]
  feature_vector = np.zeros(len(vocabulary))
  label_vector = np.zeros(len(vocabulary))

  word_to_index = {word:i for i, word in enumerate(vocabulary)}

  for word in bag_of_words:
    if word in vocabulary:
      feature_vector[word_to_index[word]] = 1

  if label in vocabulary:
    label_vector[word_to_index[label]] = 1

  final_df.loc[len(final_df)] = [feature_vector, label_vector]

  if i%10000 == 0:
    print(f"percentage completed: {100*i/context_df.shape[0]}%")

In [ ]:
# Tiny cleaning up of data.
final_df.drop(index=[i for i in range(0,5001)], inplace=True)

In [ ]:
final_df

In [ ]:
final_df['Features'][5001]

In [ ]:
# I used parquet because it is better for storing large values like this one.

final_df.to_parquet('first_quarter.parquet', engine='pyarrow', index=False, compression='snappy')

In [ ]:
# second quarter

dummy_data = {"Features": np.zeros(len(vocabulary)), "Label": np.zeros(len(vocabulary))}
final_df = pd.DataFrame(dummy_data)


for i in range(context_df.shape[0] // 4, context_df.shape[0] // 2):
  bag_of_words = context_df['context'][i].split()
  label = context_df['label'][i]
  feature_vector = np.zeros(len(vocabulary))
  label_vector = np.zeros(len(vocabulary))

  word_to_index = {word:i for i, word in enumerate(vocabulary)}

  for word in bag_of_words:
    if word in vocabulary:
      feature_vector[word_to_index[word]] = 1

  if label in vocabulary:
    label_vector[word_to_index[label]] = 1

  final_df.loc[len(final_df)] = [feature_vector, label_vector]

  if i%10000 == 0:
    print(f"percentage completed: {100*i/context_df.shape[0]}%")

In [ ]:
final_df.drop(index=[i for i in range(0,5001)], inplace=True)

In [ ]:
final_df

,Features,Label
5001,"[1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]"
5002,"[1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]"
5003,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]"
5004,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

In [ ]:
final_df.to_parquet('second_quarter.parquet', engine='pyarrow', index=False, compression='snappy')

In [ ]:
# third quarter

dummy_data = {"Features": np.zeros(len(vocabulary)), "Label": np.zeros(len(vocabulary))}
final_df = pd.DataFrame(dummy_data)


for i in range(context_df.shape[0] // 2, 3*context_df.shape[0] // 4):
  bag_of_words = context_df['context'][i].split()
  label = context_df['label'][i]
  feature_vector = np.zeros(len(vocabulary))
  label_vector = np.zeros(len(vocabulary))

  word_to_index = {word:i for i, word in enumerate(vocabulary)}

  for word in bag_of_words:
    if word in vocabulary:
      feature_vector[word_to_index[word]] = 1

  if label in vocabulary:
    label_vector[word_to_index[label]] = 1

  final_df.loc[len(final_df)] = [feature_vector, label_vector]

  if i%10000 == 0:
    print(f"percentage completed: {100*i/context_df.shape[0]}%")

In [ ]:
final_df.drop(index=[i for i in range(0,5001)], inplace=True)

In [ ]:
final_df

In [ ]:
final_df.to_parquet('third_quarter.parquet', engine='pyarrow', index=False, compression='snappy')

In [ ]:
# last quarter

dummy_data = {"Features": np.zeros(len(vocabulary)), "Label": np.zeros(len(vocabulary))}
final_df = pd.DataFrame(dummy_data)


for i in range(3 * context_df.shape[0] // 4, context_df.shape[0]):
  bag_of_words = context_df['context'][i].split()
  label = context_df['label'][i]
  feature_vector = np.zeros(len(vocabulary))
  label_vector = np.zeros(len(vocabulary))

  word_to_index = {word:i for i, word in enumerate(vocabulary)}

  for word in bag_of_words:
    if word in vocabulary:
      feature_vector[word_to_index[word]] = 1

  if label in vocabulary:
    label_vector[word_to_index[label]] = 1

  final_df.loc[len(final_df)] = [feature_vector, label_vector]

  if i%10000 == 0:
    print(f"percentage completed: {100*i/context_df.shape[0]}%")

In [ ]:
final_df.drop(index=[i for i in range(0,5001)], inplace=True)

In [ ]:
final_df

In [ ]:
final_df.to_parquet('fourth_quarter.parquet', engine='pyarrow', index=False, compression='snappy')

# Train Test Split
Now that we have the four quarters. We will be using three quarters to train the model and the last quarter as validation dataset.

In [ ]:
# Concatenating the three quarters to make the test dataset

files = ["first_quarter.parquet", "second_quarter.parquet", "third_quarter.parquet"]
output_file = "train_merged.parquet"

writer = None

for file in files:
    print(f"Processing {file}")
    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(batch_size=2048):  # batch size to that we dont crash session due to lack of RAM
        table = pa.Table.from_batches([batch])
        if writer is None:
            writer = pq.ParquetWriter(output_file, table.schema)
        writer.write_table(table)

if writer:
    writer.close()

print(f"Merged parquet saved at {output_file}")


In [ ]:
pd.read_parquet("train_merged.parquet").shape

# Training
The model is very simple. The specifications of the Word Vectors we want:
- 5000 words in vocabulary
- Each vector having 60 cells
- context size is 5
- Bag of Words model, so we wont need to worry about the order of context words


To acheive this, we use a simple ANN. The ANN has 5000 neurons in the input layer so that it can take in the One-Hot Encodings' summation as input. 60 nurons as the hidden layer. 5000 neurons as the output layer. After softmax we could compare it to the label One-Hot encodings.



In [1]:
# A few issues with newer versions, so swapped back to older versions of these libraries
!pip install torch==2.2.2 torchvision==0.17.2 numpy==1.26 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121 -qq

ERROR: Could not find a version that satisfies the requirement numpy==1.26 (from versions: 1.26.2, 1.26.3, 2.1.2)
ERROR: No matching distribution found for numpy==1.26


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

In [3]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(5000, 60)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(60, 5000)
        # softmax will automatically be applied internally

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [4]:
def train_model(parquet_file, model, optimizer, criterion, device, epochs=2, batch_size=256):
    parquet = pq.ParquetFile(parquet_file)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        running_loss = 0.0
        steps = 0

        for batch in parquet.iter_batches(batch_size=batch_size):
            df = batch.to_pandas()

            # Case 1: Data stored as wide table
            if df.shape[1] >= 10000:
                X = torch.tensor(df.iloc[:, :5000].astype("float32").to_numpy(), device=device)
                y_onehot = torch.tensor(df.iloc[:, 5000:].astype("float32").to_numpy(), device=device)

            # Case 2: Data stored as object columns with list per row
            else:
                X = torch.tensor(np.stack(df.iloc[:, 0].values), dtype=torch.float32, device=device)
                y_onehot = torch.tensor(np.stack(df.iloc[:, 1].values), dtype=torch.float32, device=device)

            # Convert one-hot to class indices
            y = torch.argmax(y_onehot, dim=1)

            # Forward + backward passes
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            steps += 1

            if steps % 20 == 0:
                print(f"Step {steps}, Loss: {running_loss/steps:.4f}")

        print(f"Epoch {epoch+1} Loss: {running_loss/steps:.4f}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ANN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_model("train.parquet", model, optimizer, criterion, device, epochs=50, batch_size=512)

Epoch 1/50
Step 20, Loss: 8.4707
Step 40, Loss: 8.4022
Step 60, Loss: 8.2524
Step 80, Loss: 8.0096
Step 100, Loss: 7.7766
Step 120, Loss: 7.5839
Step 140, Loss: 7.4172
Step 160, Loss: 7.3341
Step 180, Loss: 7.2394
Step 200, Loss: 7.1706
Step 220, Loss: 7.0990
Step 240, Loss: 7.0580
Step 260, Loss: 7.0070
Step 280, Loss: 6.9556
Step 300, Loss: 6.9028
Epoch 1 Loss: 6.8988
Epoch 2/50
Step 20, Loss: 6.0775
Step 40, Loss: 6.1988
Step 60, Loss: 6.2188
Step 80, Loss: 6.1639
Step 100, Loss: 6.1558
Step 120, Loss: 6.1659
Step 140, Loss: 6.1549
Step 160, Loss: 6.1818
Step 180, Loss: 6.1784
Step 200, Loss: 6.1790
Step 220, Loss: 6.1727
Step 240, Loss: 6.1798
Step 260, Loss: 6.1724
Step 280, Loss: 6.1609
Step 300, Loss: 6.1440
Epoch 2 Loss: 6.1451
Epoch 3/50
Step 20, Loss: 5.9102
Step 40, Loss: 6.0572
Step 60, Loss: 6.0868
Step 80, Loss: 6.0399
Step 100, Loss: 6.0300
Step 120, Loss: 6.0407
Step 140, Loss: 6.0308
Step 160, Loss: 6.0471
Step 180, Loss: 6.0421
Step 200, Loss: 6.0386
Step 220, Loss: 6

In [ ]:
# Saving checkpoint after 50 epochs(not neccessary, but i was afraid of losing data)
checkpoint = {
    "epoch": 50,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}
torch.save(checkpoint, "model_checkpoint.pth")

In [ ]:
# Continuing training with the model we had before
checkpoint = torch.load("model_checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"]
print(f"Resuming from epoch {start_epoch}")

Resuming from epoch 50


In [ ]:
# Since we already have a low loss, we could reduce learning rate to reach the minima easily and not bounce aorund
for g in optimizer.param_groups:
    g['lr'] = 1e-4 # from 1e-3

In [ ]:
train_model("train.parquet", model, optimizer, criterion, device, epochs=50, batch_size=512)

Epoch 1/50
Step 20, Loss: 2.8044
Step 40, Loss: 2.8766
Step 60, Loss: 2.9145
Step 80, Loss: 2.9293
Step 100, Loss: 2.8815
Step 120, Loss: 2.8782
Step 140, Loss: 2.8800
Step 160, Loss: 2.8242
Step 180, Loss: 2.8235
Step 200, Loss: 2.8345
Step 220, Loss: 2.8343
Step 240, Loss: 2.8024
Step 260, Loss: 2.8134
Step 280, Loss: 2.8173
Step 300, Loss: 2.8266
Epoch 1 Loss: 2.8167
Epoch 2/50
Step 20, Loss: 2.7999
Step 40, Loss: 2.8729
Step 60, Loss: 2.9123
Step 80, Loss: 2.9280
Step 100, Loss: 2.8803
Step 120, Loss: 2.8771
Step 140, Loss: 2.8787
Step 160, Loss: 2.8223
Step 180, Loss: 2.8210
Step 200, Loss: 2.8316
Step 220, Loss: 2.8312
Step 240, Loss: 2.7991
Step 260, Loss: 2.8100
Step 280, Loss: 2.8139
Step 300, Loss: 2.8233
Epoch 2 Loss: 2.8133
Epoch 3/50
Step 20, Loss: 2.7957
Step 40, Loss: 2.8688
Step 60, Loss: 2.9086
Step 80, Loss: 2.9245
Step 100, Loss: 2.8769
Step 120, Loss: 2.8738
Step 140, Loss: 2.8756
Step 160, Loss: 2.8190
Step 180, Loss: 2.8176
Step 200, Loss: 2.8281
Step 220, Loss: 2

In [ ]:
# Final save of embeddings
checkpoint = {
    "epoch": 50,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}
torch.save(checkpoint, "model_checkpoint_final.pth")

In [5]:
!pip install "sympy>=1.12"

In [6]:
# more training (in a new device because i ran out of free cloud GPU)
import torch
import torch.nn as nn
import torch.optim as optim
import pyarrow.parquet as pq
import numpy as np

model = ANN()
checkpoint = torch.load("model_checkpoint_final.pth", map_location="cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(checkpoint["model_state"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])
criterion = nn.CrossEntropyLoss()

train_model("train.parquet", model, optimizer, criterion, device, epochs=100, batch_size=512)

Epoch 1/100
Step 20, Loss: 2.6602
Step 40, Loss: 2.7320
Step 60, Loss: 2.7745
Step 80, Loss: 2.7903
Step 100, Loss: 2.7421
Step 120, Loss: 2.7415
Step 140, Loss: 2.7441
Step 160, Loss: 2.6899
Step 180, Loss: 2.6885
Step 200, Loss: 2.7000
Step 220, Loss: 2.7003
Step 240, Loss: 2.6696
Step 260, Loss: 2.6813
Step 280, Loss: 2.6861
Step 300, Loss: 2.6959
Epoch 1 Loss: 2.6859
Epoch 2/100
Step 20, Loss: 2.6580
Step 40, Loss: 2.7297
Step 60, Loss: 2.7722
Step 80, Loss: 2.7880
Step 100, Loss: 2.7397
Step 120, Loss: 2.7392
Step 140, Loss: 2.7418
Step 160, Loss: 2.6876
Step 180, Loss: 2.6862
Step 200, Loss: 2.6977
Step 220, Loss: 2.6980
Step 240, Loss: 2.6674
Step 260, Loss: 2.6791
Step 280, Loss: 2.6838
Step 300, Loss: 2.6936
Epoch 2 Loss: 2.6836
Epoch 3/100
Step 20, Loss: 2.6557
Step 40, Loss: 2.7274
Step 60, Loss: 2.7699
Step 80, Loss: 2.7857
Step 100, Loss: 2.7374
Step 120, Loss: 2.7369
Step 140, Loss: 2.7395
Step 160, Loss: 2.6854
Step 180, Loss: 2.6839
Step 200, Loss: 2.6955
Step 220, Loss

In [7]:
torch.save({
    "epoch": checkpoint["epoch"] + 10,   # continue epoch count
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
}, "model_checkpoint_200_epochs.pth")

In [8]:
# more training (continue from checkpoint)
import torch
import torch.nn as nn
import torch.optim as optim

model = ANN()
checkpoint = torch.load("model_checkpoint_200_epochs.pth", map_location="cuda" if torch.cuda.is_available() else "cpu")

# restore model + optimizer
model.load_state_dict(checkpoint["model_state"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])
criterion = nn.CrossEntropyLoss()

# train 100 more epochs
train_model("train.parquet", model, optimizer, criterion, device, epochs=100, batch_size=512)

# save new checkpoint
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}, "model_checkpoint_300_epochs.pth")


Epoch 1/100
Step 20, Loss: 2.4685
Step 40, Loss: 2.5376
Step 60, Loss: 2.5808
Step 80, Loss: 2.5936
Step 100, Loss: 2.5431
Step 120, Loss: 2.5443
Step 140, Loss: 2.5462
Step 160, Loss: 2.4947
Step 180, Loss: 2.4928
Step 200, Loss: 2.5053
Step 220, Loss: 2.5056
Step 240, Loss: 2.4760
Step 260, Loss: 2.4878
Step 280, Loss: 2.4920
Step 300, Loss: 2.5016
Epoch 1 Loss: 2.4917
Epoch 2/100
Step 20, Loss: 2.4669
Step 40, Loss: 2.5360
Step 60, Loss: 2.5791
Step 80, Loss: 2.5920
Step 100, Loss: 2.5414
Step 120, Loss: 2.5426
Step 140, Loss: 2.5445
Step 160, Loss: 2.4931
Step 180, Loss: 2.4911
Step 200, Loss: 2.5036
Step 220, Loss: 2.5039
Step 240, Loss: 2.4744
Step 260, Loss: 2.4861
Step 280, Loss: 2.4903
Step 300, Loss: 2.4999
Epoch 2 Loss: 2.4900
Epoch 3/100
Step 20, Loss: 2.4652
Step 40, Loss: 2.5343
Step 60, Loss: 2.5775
Step 80, Loss: 2.5903
Step 100, Loss: 2.5396
Step 120, Loss: 2.5409
Step 140, Loss: 2.5428
Step 160, Loss: 2.4914
Step 180, Loss: 2.4894
Step 200, Loss: 2.5019
Step 220, Loss

In [9]:
torch.save({
    "epoch": checkpoint["epoch"] + 10,   # continue epoch count
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
}, "model_checkpoint_300_epochs.pth")

In [10]:
# more training (continue from checkpoint)
import torch
import torch.nn as nn
import torch.optim as optim

model = ANN()
checkpoint = torch.load("model_checkpoint_300_epochs.pth", map_location="cuda" if torch.cuda.is_available() else "cpu")

# restore model + optimizer
model.load_state_dict(checkpoint["model_state"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])
criterion = nn.CrossEntropyLoss()

# train 100 more epochs
train_model("train.parquet", model, optimizer, criterion, device, epochs=100, batch_size=512)



Epoch 1/100
Step 20, Loss: 2.3251
Step 40, Loss: 2.3913
Step 60, Loss: 2.4340
Step 80, Loss: 2.4441
Step 100, Loss: 2.3916
Step 120, Loss: 2.3937
Step 140, Loss: 2.3945
Step 160, Loss: 2.3451
Step 180, Loss: 2.3426
Step 200, Loss: 2.3555
Step 220, Loss: 2.3558
Step 240, Loss: 2.3271
Step 260, Loss: 2.3385
Step 280, Loss: 2.3420
Step 300, Loss: 2.3512
Epoch 1 Loss: 2.3416
Epoch 2/100
Step 20, Loss: 2.3238
Step 40, Loss: 2.3900
Step 60, Loss: 2.4327
Step 80, Loss: 2.4428
Step 100, Loss: 2.3903
Step 120, Loss: 2.3924
Step 140, Loss: 2.3931
Step 160, Loss: 2.3437
Step 180, Loss: 2.3413
Step 200, Loss: 2.3542
Step 220, Loss: 2.3545
Step 240, Loss: 2.3258
Step 260, Loss: 2.3372
Step 280, Loss: 2.3407
Step 300, Loss: 2.3499
Epoch 2 Loss: 2.3403
Epoch 3/100
Step 20, Loss: 2.3226
Step 40, Loss: 2.3887
Step 60, Loss: 2.4314
Step 80, Loss: 2.4415
Step 100, Loss: 2.3889
Step 120, Loss: 2.3911
Step 140, Loss: 2.3918
Step 160, Loss: 2.3424
Step 180, Loss: 2.3400
Step 200, Loss: 2.3529
Step 220, Loss

In [11]:
# save new checkpoint
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}, "model_checkpoint_400_epochs.pth")


In [12]:
# more training (continue from checkpoint)
import torch
import torch.nn as nn
import torch.optim as optim

model = ANN()
checkpoint = torch.load("model_checkpoint_400_epochs.pth", map_location="cuda" if torch.cuda.is_available() else "cpu")

# restore model + optimizer
model.load_state_dict(checkpoint["model_state"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])
criterion = nn.CrossEntropyLoss()

# train 100 more epochs
train_model("train.parquet", model, optimizer, criterion, device, epochs=100, batch_size=512)



Epoch 1/100
Step 20, Loss: 2.2116
Step 40, Loss: 2.2744
Step 60, Loss: 2.3160
Step 80, Loss: 2.3240
Step 100, Loss: 2.2701
Step 120, Loss: 2.2729
Step 140, Loss: 2.2724
Step 160, Loss: 2.2246
Step 180, Loss: 2.2218
Step 200, Loss: 2.2350
Step 220, Loss: 2.2352
Step 240, Loss: 2.2073
Step 260, Loss: 2.2184
Step 280, Loss: 2.2212
Step 300, Loss: 2.2301
Epoch 1 Loss: 2.2207
Epoch 2/100
Step 20, Loss: 2.2106
Step 40, Loss: 2.2733
Step 60, Loss: 2.3149
Step 80, Loss: 2.3229
Step 100, Loss: 2.2691
Step 120, Loss: 2.2718
Step 140, Loss: 2.2713
Step 160, Loss: 2.2235
Step 180, Loss: 2.2208
Step 200, Loss: 2.2339
Step 220, Loss: 2.2341
Step 240, Loss: 2.2063
Step 260, Loss: 2.2173
Step 280, Loss: 2.2201
Step 300, Loss: 2.2290
Epoch 2 Loss: 2.2196
Epoch 3/100
Step 20, Loss: 2.2095
Step 40, Loss: 2.2722
Step 60, Loss: 2.3139
Step 80, Loss: 2.3218
Step 100, Loss: 2.2680
Step 120, Loss: 2.2707
Step 140, Loss: 2.2702
Step 160, Loss: 2.2224
Step 180, Loss: 2.2197
Step 200, Loss: 2.2328
Step 220, Loss

In [13]:
# save new checkpoint
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}, "model_checkpoint_500_epochs.pth")


# Model
This is decent for a self trained model. The State of art models have a loss of 1 after training. For a self trained model, loss of 2-3 is decent and will be able to tell the similarities between most words.

Also keep in mind that most embeddings are significantly larger going upto 1000 embedding dimensions. Our model has only 60 embedding dimensions.

In [15]:
checkpoint = torch.load("model_checkpoint_500_epochs.pth", map_location="cuda" if torch.cuda.is_available() else "cpu")

model = ANN()
model.load_state_dict(checkpoint["model_state"])  # use the right key
model.eval()


ANN(
  (fc1): Linear(in_features=5000, out_features=60, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=60, out_features=5000, bias=True)
)

In [17]:
embedding_matrix = model.fc2.weight.data.clone().cpu().numpy()

In [18]:
embedding_matrix.shape

(5000, 60)

# Extracting Vectors
Now that we have the Weights of the trained model. We can go ahead and extract the Vectors for each word in our vocabulary.

All the weights leading up to each of the neurons from the hidden layer when arranged in order make up the word vector of that word.

In [19]:
df_embeddings = pd.DataFrame(embedding_matrix)

In [20]:
df_embeddings.head()

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.035566,-0.280325,-0.209905,0.092727,-0.512841,-0.070271,0.017452,-0.073750,-0.246000,-0.034664,...,-0.229457,0.116644,-0.187131,-0.380852,-0.201888,-0.175198,0.051587,-0.059986,0.081284,-0.065884
1,-0.366226,0.445310,-0.370340,-0.565965,0.480987,0.388533,-0.032340,-0.365572,0.032613,0.283541,...,-0.502574,-0.020204,-0.924985,-0.322606,-0.402044,0.259014,0.882215,-0.163126,-0.626576,-0.516701
2,0.258176,-0.317606,0.196751,0.285550,-0.025030,-0.829887,-0.403331,0.248189,-0.258676,0.206602,...,-1.072994,0.137963,-0.395314,0.433717,-0.032723,0.322714,0.758324,0.188737,0.398276,-1.168513
3,0.498756,0.103553,-0.771941,-0.639479,0.631630,0.101978,-0.631072,-0.583583,0.081321,-0.668437,...,0.117473,-1.194255,-0.275528,-0.327657,-0.742398,0.609596,-0.027341,0.566738,0.826060,-0.184296
4,-0.446127,0.729279,-0.294486,0.385511,0.426112,-0.633729,0.385096,0.600576,0.476241,0.169516,...,-1.250619,-0.297400,-0.017004,-0.843019,-0.027971,-0.229272,0.085112,0.341840,-0.174408,0.065009


In [30]:
# Labeling all the word vectors with their respective word
df_embeddings.insert(0, "word", vocabulary)

In [31]:
# Tweaking a few settings to display the embeddings better
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 6)

print(df_embeddings.head(10))

   word         0         1         2         3         4         5         6         7         8         9        10        11        12        13        14        15        16        17        18        19        20        21        22        23        24        25        26        27        28        29        30        31        32        33        34        35        36        37        38        39        40        41        42        43        44        45        46        47        48        49        50        51        52        53        54        55        56        57        58        59
0   the  0.035566 -0.280325 -0.209905  0.092727 -0.512841 -0.070271  0.017452 -0.073750 -0.246000 -0.034664 -0.240868  0.028574 -0.095689 -0.298459 -0.177851 -0.097263 -0.297415  0.075913  0.372466 -0.034589 -0.123280 -0.149911 -0.133922 -0.085907  0.010811 -0.168051  0.136708 -0.083664  0.012976 -0.026008 -0.056302 -0.039275  0.056458 -0.127233 -0.252066 -0.004904 -0.250311 -0.123614 -0.0

In [32]:
# Saving the model
pd.to_pickle(df_embeddings, "embeddings_final.pkl")

# Evaluation

We can now see if our very primitive model can capture relationships between words based off their ouccerence.

We could do this in multiple ways:
- Running a KNN algorithm to see which words are closer
- Running similarity tests based off Cosine Similarity


We opt for the second option because KNNs hallucinate when the number of dimensions are too high.

In [33]:
df_embeddings = pd.read_pickle('embeddings_final.pkl')

In [34]:
import torch
import torch.nn.functional as F

def get_vector(word, df):
    row = df[df["word"] == word]
    if row.empty:
        raise ValueError(f"Word '{word}' not in vocabulary")
    vec = torch.tensor(row.iloc[:, 1:].values, dtype=torch.float32).squeeze(0)
    return vec

def most_similar(word, df, topn=10):
    target_vec = get_vector(word, df)
    all_words = df["word"].values
    all_vecs = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)

    # Compute cosine similarity with all words
    sims = F.cosine_similarity(target_vec.unsqueeze(0), all_vecs)

    # Sort and pick top-N (skip self at index of the word itself)
    topk = torch.topk(sims, topn + 1)  # +1 because we include the word itself
    results = []
    for idx, score in zip(topk.indices, topk.values):
        w = all_words[idx]
        if w != word:  # skip the same word
            results.append((w, score.item()))
        if len(results) == topn:
            break

    return results



In [38]:
word = "war"
neighbors = most_similar(word, df_embeddings, topn=2)
print(f"Most similar to '{word}':")
for w, s in neighbors:
    print(f"{w}: {s:.4f}")


Most similar to 'war':
vichy: 0.6170
conflict: 0.6149


In [37]:
word = "music"
neighbors = most_similar(word, df_embeddings, topn=2)
print(f"Most similar to '{word}':")
for w, s in neighbors:
    print(f"{w}: {s:.4f}")


Most similar to 'music':
soundtrack: 0.6731
aerial: 0.6603


In [51]:
word = "london"
neighbors = most_similar(word, df_embeddings, topn=3)
print(f"Most similar to '{word}':")
for w, s in neighbors:
    print(f"{w}: {s:.4f}")


Most similar to 'london':
hampton: 0.5774
community: 0.5736
connecticut: 0.5712
